# Econometria II - Modelagem de Volatilidade (ARCH/GARCH)
## Aplicacao aos retornos diarios do S&P 500
**2025.1 - IBMEC Rio**

Notebook dividido em secoes. Antes de cada bloco de codigo ha uma celula de comentario explicando a acao.

**Como rodar:** no Colab, *Ambiente de execucao -> Executar tudo* (Ctrl+F9). A internet do Colab ja vem liberada.

**Roteiro:** 0. Setup | 1. Simulacao ARCH(1) | 2. Dados S&P 500 | 3. Retornos e descritiva | 4. ADF | 5. Selecao AIC/BIC | 6. Estimacao GARCH(1,1) | 7. Diagnosticos | 8. Previsao

## 0. Instalacao e importacao dos pacotes

**Acao:** instala as bibliotecas. `arch` (GARCH), `yfinance`/`pandas_datareader` (dados), `statsmodels` (testes).

In [ ]:
!pip install arch yfinance pandas_datareader statsmodels -q

**Acao:** importa modulos e fixa a semente para reprodutibilidade da simulacao.

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
from scipy import stats
warnings.filterwarnings("ignore")
np.random.seed(42)

## 1. Simulacao de um processo ARCH(1)
$$\epsilon_t=\sqrt{h_t}\,z_t,\quad h_t=\alpha_0+\alpha_1\epsilon_{t-1}^2,\quad z_t\sim N(0,1)$$
Estacionariedade exige $\alpha_1<1$; variancia incondicional $=\alpha_0/(1-\alpha_1)$.

**Acao:** define parametros e vetores da simulacao.

In [ ]:
T = 1000
alpha0, alpha1 = 0.1, 0.8
epsilon = np.zeros(T); h = np.zeros(T)
h[0] = alpha0 / (1 - alpha1)

**Acao:** gera os choques e itera a recursao do ARCH(1).

In [ ]:
z = np.random.normal(size=T)
epsilon[0] = np.sqrt(h[0]) * z[0]
for t in range(1, T):
    h[t] = alpha0 + alpha1 * (epsilon[t-1] ** 2)
    epsilon[t] = np.sqrt(h[t]) * z[t]

**Acao:** plota a serie simulada (note os clusters de volatilidade).

In [ ]:
plt.figure()
plt.plot(epsilon, linewidth=0.8)
plt.title(r"Serie simulada $\epsilon_t$ - ARCH(1)")
plt.xlabel("Periodo"); plt.ylabel(r"$\epsilon_t$"); plt.axhline(0, color="black", linewidth=0.8)
plt.show()

**Acao:** plota a variancia condicional $h_t$.

In [ ]:
plt.figure()
plt.plot(h, color="orange", linewidth=0.8)
plt.title(r"Variancia condicional $h_t$ - ARCH(1)")
plt.xlabel("Periodo"); plt.ylabel(r"$h_t$")
plt.show()

## 2. Download dos precos diarios do S&P 500

**Acao:** define a janela e uma funcao que baixa o preco tentando yfinance, depois Stooq, depois FRED.

In [ ]:
DATA_INICIO = "2015-01-01"
DATA_FIM    = "2024-12-31"

def baixar_sp500(inicio=DATA_INICIO, fim=DATA_FIM):
    try:
        import yfinance as yf
        df = yf.download("^GSPC", start=inicio, end=fim, progress=False, auto_adjust=False)
        if df is not None and len(df) > 0:
            preco = df["Close"]
            if isinstance(preco, pd.DataFrame): preco = preco.iloc[:, 0]
            preco = preco.dropna(); preco.name = "SP500"
            print(f"[dados] yfinance | {len(preco)} obs "
                  f"({preco.index.min().date()} a {preco.index.max().date()})")
            return preco
    except Exception as e:
        print("[dados] yfinance indisponivel:", e)
    try:
        import pandas_datareader.data as web
        df = web.DataReader("^SPX", "stooq", inicio, fim).sort_index()
        if len(df) > 0:
            preco = df["Close"].dropna(); preco.name = "SP500"
            print(f"[dados] Stooq | {len(preco)} obs"); return preco
    except Exception as e:
        print("[dados] Stooq indisponivel:", e)
    try:
        import pandas_datareader.data as web
        df = web.DataReader("SP500", "fred", inicio, fim)
        preco = df["SP500"].dropna(); preco.name = "SP500"
        print(f"[dados] FRED | {len(preco)} obs"); return preco
    except Exception as e:
        print("[dados] FRED indisponivel:", e)
    raise RuntimeError("Nao foi possivel baixar os dados do S&P 500.")

**Acao:** executa o download e confere as ultimas linhas.

In [ ]:
sp500 = baixar_sp500()
sp500.tail()

**Acao:** plota o nivel de preco do S&P 500.

In [ ]:
plt.figure()
plt.plot(sp500, color="navy", linewidth=0.9)
plt.title("Niveis diarios do Indice S&P 500")
plt.xlabel("Data"); plt.ylabel("Preco de fechamento")
plt.show()

## 3. Retornos logaritmicos e analise descritiva

**Acao:** calcula retornos logaritmicos diarios e a volatilidade movel de 20 dias.

In [ ]:
log_return = np.log(sp500 / sp500.shift(1))
returns = log_return.dropna()
vol_20 = returns.rolling(window=20).std()

**Acao:** monta a tabela de estatisticas descritivas (curtose alta = caudas pesadas, motiva o GARCH).

In [ ]:
desc = pd.DataFrame({
    "Estatistica": ["N. observacoes","Media","Desvio-padrao","Minimo","Maximo","Assimetria","Curtose"],
    "Valor": [int(returns.count()), returns.mean(), returns.std(), returns.min(),
              returns.max(), stats.skew(returns), stats.kurtosis(returns, fisher=False)],
})
print(desc.to_string(index=False))

**Acao:** plota os retornos diarios.

In [ ]:
plt.figure()
plt.plot(returns, linewidth=0.7)
plt.title("Retornos logaritmicos diarios - S&P 500")
plt.xlabel("Data"); plt.ylabel("Retorno logaritmico"); plt.axhline(0, color="black", linewidth=0.8)
plt.show()

**Acao:** plota a volatilidade movel de 20 dias.

In [ ]:
plt.figure()
plt.plot(vol_20, color="red", linewidth=0.8)
plt.title("Volatilidade movel de 20 dias - S&P 500")
plt.xlabel("Data"); plt.ylabel("Desvio-padrao (20 dias)")
plt.show()

## 4. Teste de raiz unitaria (Dickey-Fuller Aumentado - ADF)
H0 = raiz unitaria. Esperado: preco nao estacionario, retorno estacionario.

**Acao:** funcao que roda o ADF e imprime a conclusao.

In [ ]:
def reporta_adf(serie, nome):
    adf_stat, p, lags, nobs, crit, _ = adfuller(serie.dropna(), autolag="AIC")
    print(f"ADF - {nome}: estatistica={adf_stat:.3f} | p-valor={p:.4f} | lags={lags}")
    print("  -> ESTACIONARIA." if p < 0.05 else "  -> RAIZ UNITARIA (nao estacionaria).")

**Acao:** aplica o ADF ao preco e aos retornos.

In [ ]:
reporta_adf(sp500, "Preco do S&P 500 (nivel)")
reporta_adf(returns, "Retornos logaritmicos")

## 5. Selecao de modelo por AIC / BIC
Escolhe-se a especificacao de menores criterios de informacao.

**Acao:** escala os retornos para porcentagem (x100), recomendado pelo pacote `arch`.

In [ ]:
returns_pct = returns * 100

**Acao:** estima varias especificacoes e compara por AIC/BIC.

In [ ]:
espec = [("ARCH(1)",1,0),("GARCH(1,1)",1,1),("GARCH(1,2)",1,2),("GARCH(2,1)",2,1)]
linhas = []
for nome,p,q in espec:
    r = arch_model(returns_pct, mean="Constant", vol="GARCH", p=p, q=q).fit(disp="off")
    linhas.append((nome, r.aic, r.bic))
comp = pd.DataFrame(linhas, columns=["Modelo","AIC","BIC"])
print(comp.to_string(index=False))
print("\nMenor BIC:", comp.loc[comp["BIC"].idxmin(),"Modelo"])

## 6. Estimacao do modelo GARCH(1,1)
$h_t=\alpha_0+\alpha_1\epsilon_{t-1}^2+\beta_1 h_{t-1}$, por maxima verossimilhanca.

**Acao:** ajusta o GARCH(1,1) e imprime o sumario com coeficientes e p-valores.

In [ ]:
model = arch_model(returns_pct, mean="Constant", vol="GARCH", p=1, q=1)
result = model.fit(disp="off")
print(result.summary())

## 7. Diagnosticos do modelo

**Acao:** compara a volatilidade condicional do GARCH com a historica de 20 dias.

In [ ]:
cond_vol = pd.Series(result.conditional_volatility, index=returns_pct.index)
hist_vol_20 = vol_20 * 100
plt.figure()
plt.plot(cond_vol, label="Condicional (GARCH)", linewidth=0.9)
plt.plot(hist_vol_20, label="Historica (20 dias)", alpha=0.7, linewidth=0.9)
plt.title("Volatilidade condicional (GARCH) vs. historica (20 dias)")
plt.xlabel("Data"); plt.ylabel("Volatilidade (%)"); plt.legend()
plt.show()

**Acao:** calcula e plota os residuos padronizados.

In [ ]:
std_resid = pd.Series(result.resid / result.conditional_volatility, index=returns_pct.index)
plt.figure()
plt.plot(std_resid, linewidth=0.7)
plt.title("Residuos padronizados do GARCH(1,1)")
plt.xlabel("Data"); plt.ylabel("Residuo padronizado"); plt.axhline(0, color="black", linewidth=0.8)
plt.show()

**Acao:** ACF dos residuos padronizados (verifica autocorrelacao na media).

In [ ]:
plt.figure()
plot_acf(std_resid, ax=plt.gca(), lags=20, zero=False)
plt.title("ACF dos residuos padronizados")
plt.xlabel("Defasagem (lag)"); plt.ylabel("Autocorrelacao")
plt.show()

**Acao:** ACF dos residuos ao quadrado (teste visual chave da dinamica da variancia).

In [ ]:
plt.figure()
plot_acf(std_resid**2, ax=plt.gca(), lags=20, zero=False)
plt.title("ACF dos quadrados dos residuos padronizados")
plt.xlabel("Defasagem (lag)"); plt.ylabel("Autocorrelacao")
plt.show()

**Acao:** grafico Q-Q dos residuos contra a normal.

In [ ]:
plt.figure()
sm.qqplot(std_resid, line="s", ax=plt.gca())
plt.title("Grafico Q-Q dos residuos padronizados")
plt.show()

**Acao:** persistencia e teste de Ljung-Box nos residuos ao quadrado.

In [ ]:
alpha1_hat = result.params.get("alpha[1]", np.nan)
beta1_hat  = result.params.get("beta[1]", np.nan)
print(f"Persistencia (alpha1 + beta1) = {alpha1_hat + beta1_hat:.4f}")
lb = acorr_ljungbox(std_resid**2, lags=[10], return_df=True)
stat, p = lb["lb_stat"].iloc[0], lb["lb_pvalue"].iloc[0]
print(f"Ljung-Box(10) nos residuos^2: stat={stat:.3f} | p-valor={p:.4f}")
print("-> Bom ajuste (sem autocorrelacao residual)." if p > 0.05 else "-> Ajuste insuficiente.")

## 8. Previsao de volatilidade
Projeta a variancia condicional para os proximos dias a partir do ultimo estado estimado.

**Acao:** gera a previsao de 20 dias, converte para volatilidade (%) e plota.

In [ ]:
HORIZONTE = 20
forecast = result.forecast(horizon=HORIZONTE, reindex=False)
vol_prevista = np.sqrt(forecast.variance.values[-1, :])
plt.figure()
plt.plot(range(1, HORIZONTE+1), vol_prevista, marker="o")
plt.title(f"Previsao da volatilidade condicional - proximos {HORIZONTE} dias")
plt.xlabel("Dias a frente"); plt.ylabel("Volatilidade prevista (%)"); plt.grid(alpha=0.3)
plt.show()
print(f"Dia 1: {vol_prevista[0]:.3f}% | Dia 5: {vol_prevista[4]:.3f}% | Dia 20: {vol_prevista[-1]:.3f}%")